# DPO Fine-Tuning: PocketCoder 100M on `jondurbin/py-dpo-v0.1`

This is step 4 of your pipeline (syntax pretrain → distillation → broad SFT → **DPO polish**).

Your model (`GPT` / `GPTConfig`) is a **custom nanoGPT-style architecture**, not a native
`transformers` model, so `trl.DPOTrainer` won't work with it out of the box (it expects
`AutoModelForCausalLM`-style models). This notebook implements the **DPO loss manually**,
using your exact model class, so it plugs straight into your existing SFT checkpoint.

**What DPO does here:** for every `(prompt, chosen, rejected)` triple, it nudges your policy
model to assign *relatively* higher log-probability to `chosen` than to `rejected`, compared to
a frozen copy of your SFT model (the "reference" model). It does **not** need a reward model.

**Steps:**
1. Install packages
2. Config — fill in your SFT repo + output repo
3. Load `jondurbin/py-dpo-v0.1` (prompt / chosen / rejected / id)
4. Rebuild your exact `GPT` architecture
5. Load the SFT checkpoint twice: once as the trainable **policy**, once as the frozen **reference**
6. Tokenize triples using the same `### Problem / ### Solution` template as your SFT run
7. Manual DPO loss + training loop
8. Push the DPO model to the Hub


In [ ]:
!pip install -q -U datasets transformers huggingface_hub safetensors tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 129.1 MB/s eta 0:00:00


## 1. Config — fill these in

In [ ]:
# ======================= FILL THESE IN =======================
HF_SFT_REPO  = "Ananda100/100m-sftd-python"     # your SFT model from the previous notebook
HF_DPO_REPO  = "Ananda100/pocketcoder-dpo"     # where the DPO model will be pushed
HF_TOKEN     = ""                               # your HF token (needs write access for the final push)
# ===============================================================
HF_DPO_DATASET_REPO = "Ananda100/py-dpo-code-only"   # prompt / chosen / rejected / id
DATASET_SPLIT = "train"

# DPO hyperparameters
BETA = 0.1                 # how strongly to trust the reference model (0.1-0.3 is typical)
LEARNING_RATE = 5e-6        # DPO wants a MUCH lower LR than SFT - you're nudging, not re-teaching
NUM_EPOCHS = 2              # DPO datasets are small and easy to overfit/mode-collapse on - start with 1
MICRO_BATCH = 1             # forward pass batch (kept at 1 to sidestep padding - see note below)
GRAD_ACCUM_PAIRS = 16       # effective batch = MICRO_BATCH * GRAD_ACCUM_PAIRS preference pairs
MAX_GRAD_NORM = 1.0
EVAL_EVERY_STEPS = 50
HELD_OUT_FRACTION = 0.02    # small val slice to track DPO accuracy/margin, mirrors your SFT split
SEED = 42

# Note on batching: your GPT class has no attention-mask / padding support (it always assumes a
# dense, unpadded sequence with is_causal=True). Padding variable-length prompt+completion pairs
# would silently corrupt the reference-model logprobs unless you also add a padding mask to
# CausalSelfAttention. Rather than touch your architecture, this notebook processes one pair at a
# time and accumulates gradients - correct, and plenty fast for a 100M model on ~9.5k pairs.

## 2. Load the DPO dataset from the Hugging Face Hub

In [ ]:
from datasets import load_dataset

print(f"Loading dataset from Hugging Face Hub: {HF_DPO_DATASET_REPO} (split={DATASET_SPLIT})")
dpo_ds = load_dataset(HF_DPO_DATASET_REPO, split=DATASET_SPLIT, token=HF_TOKEN or None)
print(dpo_ds)
print("Columns:", dpo_ds.column_names)

assert {"prompt", "chosen", "rejected"}.issubset(set(dpo_ds.column_names)), \
    f"Expected prompt/chosen/rejected columns, got {dpo_ds.column_names}"


Loading dataset from Hugging Face Hub: Ananda100/py-dpo-code-only (split=train)


README.md:   0%|          | 0.00/379 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.30MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6945 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'chosen', 'rejected'],
    num_rows: 6945
})
Columns: ['id', 'prompt', 'chosen', 'rejected']


## 3. Train / val split

Same idea as your SFT notebook: hold out a small slice purely to *monitor* DPO accuracy
(how often the policy prefers `chosen` over `rejected`) and reward margin during training.
This is not the same as the pass@1 functional test - DPO doesn't need one, since it isn't
teaching new capability, just re-weighting preferences the SFT stage already knows how to produce.

In [ ]:
import random

records = dpo_ds.to_list()
if not records or "id" not in records[0] or any(r.get("id") is None for r in records):
    for i, r in enumerate(records):
        r["id"] = i

random.seed(SEED)
random.shuffle(records)
n_val = max(1, int(len(records) * HELD_OUT_FRACTION))
val_records = records[:n_val]
train_records = records[n_val:]
print(f"DPO pairs -> train: {len(train_records)}, val: {len(val_records)}")


DPO pairs -> train: 6807, val: 138


## 4. Rebuild the exact model architecture

Identical to your SFT notebook - copied verbatim so the checkpoint loads cleanly.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(torch.nn.functional, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 32256
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        dev = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"sequence length {t} exceeds block_size {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=dev)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    def full_logits(self, idx):
        # Like forward(), but always returns logits for every position (needed for DPO
        # sequence log-probs) instead of only the last token when targets is None.
        dev = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=dev)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        return self.lm_head(x)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_token_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            if eos_token_id is not None and idx.size(0) == 1 and idx_next.item() == eos_token_id:
                break
        return idx


## 5. Load the SFT checkpoint twice — policy (trainable) + reference (frozen)

This is the DPO-specific part: the **reference model** is a frozen snapshot of your SFT model.
It never gets gradient updates. It exists purely so the loss can measure *how much* the policy's
preference for `chosen` over `rejected` has shifted relative to where it started - this is what
keeps DPO from just collapsing into "always output whatever code is in `chosen`, verbatim".

In [ ]:
import copy, json as _json
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

config_path = hf_hub_download(repo_id=HF_SFT_REPO, filename="config.json", token=HF_TOKEN or None)
weights_path = hf_hub_download(repo_id=HF_SFT_REPO, filename="model.safetensors", token=HF_TOKEN or None)

with open(config_path) as f:
    cfg_dict = _json.load(f)
config = GPTConfig(**cfg_dict)

policy_model = GPT(config)
policy_model.load_state_dict(load_file(weights_path))
policy_model.to(device)
for p in policy_model.parameters():
    p.requires_grad = True
policy_model.train()

ref_model = copy.deepcopy(policy_model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(HF_SFT_REPO, token=HF_TOKEN or None)
if tokenizer.eos_token_id is None:
    raise ValueError("Tokenizer has no eos_token_id set - fix before tokenizing.")

n_params = sum(p.numel() for p in policy_model.parameters()) / 1e6
print(f"Loaded SFT model from {HF_SFT_REPO} - {n_params:.2f}M params (policy trainable, reference frozen)")


Loaded SFT model from Ananda100/100m-sftd-python - 95.87M params (policy trainable, reference frozen)


## 6. Tokenize (prompt, chosen, rejected) triples

Uses the **same template your SFT run used**, so the DPO stage is reinforcing preferences
inside the exact distribution the model already learned to produce, rather than introducing a
new prompt format:

```
### Problem
{prompt}

### Solution
```python
{completion}
```
```

Each triple is truncated to `block_size`. If the prompt alone doesn't leave room for at least a
few completion tokens, the pair is skipped (this is rare with a 512-token block size, but the
`py-dpo-v0.1` dataset does have some long problems).

In [ ]:
def format_prompt(problem):
    return f"### Problem\n{problem}\n\n### Solution\n```python\n"

def format_completion(text):
    return f"{text}\n```"

def tokenize_pair(record):
    prompt_ids = tokenizer.encode(format_prompt(record["prompt"]), add_special_tokens=False)
    if len(prompt_ids) >= config.block_size - 4:
        return None  # not enough room left for any completion tokens

    chosen_ids = prompt_ids + tokenizer.encode(format_completion(record["chosen"]), add_special_tokens=False) + [tokenizer.eos_token_id]
    rejected_ids = prompt_ids + tokenizer.encode(format_completion(record["rejected"]), add_special_tokens=False) + [tokenizer.eos_token_id]

    chosen_ids = chosen_ids[:config.block_size]
    rejected_ids = rejected_ids[:config.block_size]

    # Degenerate cases: truncation ate the whole completion, or chosen==rejected after truncation
    if len(chosen_ids) <= len(prompt_ids) + 1 or len(rejected_ids) <= len(prompt_ids) + 1:
        return None
    if chosen_ids == rejected_ids:
        return None

    return {
        "prompt_len": len(prompt_ids),
        "chosen_ids": torch.tensor(chosen_ids, dtype=torch.long),
        "rejected_ids": torch.tensor(rejected_ids, dtype=torch.long),
    }

def tokenize_all(records):
    out = []
    skipped = 0
    for r in tqdm(records):
        t = tokenize_pair(r)
        if t is None:
            skipped += 1
        else:
            out.append(t)
    print(f"Tokenized {len(out)} pairs, skipped {skipped}")
    return out

from tqdm.auto import tqdm
train_pairs = tokenize_all(train_records)
val_pairs = tokenize_all(val_records)


  0%|          | 0/6807 [00:00<?, ?it/s]

Tokenized 6807 pairs, skipped 0


  0%|          | 0/138 [00:00<?, ?it/s]

Tokenized 138 pairs, skipped 0


## 7. DPO loss

Standard DPO objective (Rafailov et al., 2023):

```
loss = -log sigmoid( beta * [ (logp_chosen_policy - logp_chosen_ref)
                              - (logp_rejected_policy - logp_rejected_ref) ] )
```

`sequence_logprob` sums log-probabilities **only over the completion tokens** (prompt tokens are
masked out) - DPO cares about how likely the model is to generate `chosen` vs `rejected` given
the prompt, not the prompt itself.

In [ ]:
def sequence_logprob(model, ids, prompt_len):
    # ids: (T,) full prompt+completion+eos token ids, on CPU or device
    ids = ids.unsqueeze(0).to(device)              # (1, T)
    inputs = ids[:, :-1]
    targets = ids[:, 1:]

    logits = model.full_logits(inputs)             # (1, T-1, vocab)
    logprobs = F.log_softmax(logits, dim=-1)
    token_logprobs = logprobs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)  # (1, T-1)

    # targets[i] is the token originally at position i+1, so the completion
    # (original position >= prompt_len) starts at i = prompt_len - 1
    T_minus_1 = token_logprobs.size(1)
    mask = torch.arange(T_minus_1, device=device) >= (prompt_len - 1)
    return (token_logprobs * mask).sum()

def dpo_loss_and_metrics(pair, beta=BETA):
    policy_chosen = sequence_logprob(policy_model, pair["chosen_ids"], pair["prompt_len"])
    policy_rejected = sequence_logprob(policy_model, pair["rejected_ids"], pair["prompt_len"])
    with torch.no_grad():
        ref_chosen = sequence_logprob(ref_model, pair["chosen_ids"], pair["prompt_len"])
        ref_rejected = sequence_logprob(ref_model, pair["rejected_ids"], pair["prompt_len"])

    pi_logratios = policy_chosen - policy_rejected
    ref_logratios = ref_chosen - ref_rejected
    logits = pi_logratios - ref_logratios

    loss = -F.logsigmoid(beta * logits)
    with torch.no_grad():
        accuracy = (logits > 0).float()
        reward_margin = beta * logits
    return loss, accuracy, reward_margin

@torch.no_grad()
def evaluate_val(pairs, n=None):
    policy_model.eval()
    sample = pairs if n is None else pairs[:n]
    losses, accs, margins = [], [], []
    for pair in sample:
        loss, acc, margin = dpo_loss_and_metrics(pair)
        losses.append(loss.item()); accs.append(acc.item()); margins.append(margin.item())
    policy_model.train()
    if not losses:
        return {"loss": float("nan"), "accuracy": float("nan"), "margin": float("nan")}
    return {
        "loss": sum(losses) / len(losses),
        "accuracy": sum(accs) / len(accs),
        "margin": sum(margins) / len(margins),
    }


## 8. Training loop

Low LR, short warmup, `MICRO_BATCH=1` with gradient accumulation over `GRAD_ACCUM_PAIRS`
preference pairs per optimizer step. Keeps the best checkpoint by **val DPO accuracy**
(fraction of held-out pairs where the policy now prefers `chosen`), not val loss - loss can
keep dropping past the point where accuracy plateaus, which is an early sign of overfitting to
this specific `rejected` distribution.

In [ ]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

torch.manual_seed(SEED)
random.seed(SEED)

total_steps = max(1, (len(train_pairs) * NUM_EPOCHS) // GRAD_ACCUM_PAIRS)
warmup_steps = max(1, min(20, total_steps // 10))

optimizer = torch.optim.AdamW(policy_model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=0.0, eps=1e-9)
scheduler_warmup = LinearLR(optimizer, total_iters=warmup_steps)
scheduler_decay = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps), eta_min=LEARNING_RATE / 10)
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps])

print(f"Training pairs: {len(train_pairs)} | optimizer steps: {total_steps} | warmup: {warmup_steps}")


Training pairs: 6807 | optimizer steps: 850 | warmup: 20


In [ ]:
best_val_acc = -1.0
best_model_path = "best_dpo_model.pt"
step = 0
optimizer.zero_grad(set_to_none=True)

policy_model.train()
epoch_order = list(range(len(train_pairs)))

for epoch in range(NUM_EPOCHS):
    random.shuffle(epoch_order)
    running_loss = 0.0
    for i, idx in enumerate(tqdm(epoch_order, desc=f"epoch {epoch+1}/{NUM_EPOCHS}")):
        pair = train_pairs[idx]
        loss, acc, margin = dpo_loss_and_metrics(pair)
        (loss / GRAD_ACCUM_PAIRS).backward()
        running_loss += loss.item()

        if (i + 1) % GRAD_ACCUM_PAIRS == 0 or (i + 1) == len(epoch_order):
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            step += 1

            if step % EVAL_EVERY_STEPS == 0:
                val_metrics = evaluate_val(val_pairs)
                avg_train_loss = running_loss / (i + 1)
                print(f"step {step}: train_loss {avg_train_loss:.4f} | "
                      f"val_loss {val_metrics['loss']:.4f} | val_acc {val_metrics['accuracy']:.1%} | "
                      f"val_margin {val_metrics['margin']:.4f}")
                if val_metrics["accuracy"] > best_val_acc:
                    best_val_acc = val_metrics["accuracy"]
                    torch.save(policy_model.state_dict(), best_model_path)

final_val_metrics = evaluate_val(val_pairs)
torch.save(policy_model.state_dict(), "last_dpo_model.pt")
if best_val_acc < 0:
    best_val_acc = final_val_metrics["accuracy"]
    torch.save(policy_model.state_dict(), best_model_path)

print("--- DPO training complete ---")
print(f"Final val accuracy: {final_val_metrics['accuracy']:.1%}")
print(f"Best val accuracy seen: {best_val_acc:.1%}")


epoch 1/2:   0%|          | 0/6807 [00:00<?, ?it/s]

step 50: train_loss 0.6760 | val_loss 0.5068 | val_acc 79.7% | val_margin 0.5944
step 100: train_loss 0.6011 | val_loss 0.4261 | val_acc 81.9% | val_margin 1.1189
step 150: train_loss 0.5570 | val_loss 0.3916 | val_acc 79.0% | val_margin 1.4925
step 200: train_loss 0.5266 | val_loss 0.3636 | val_acc 83.3% | val_margin 1.6868
step 250: train_loss 0.5122 | val_loss 0.3336 | val_acc 86.2% | val_margin 1.6762
step 300: train_loss 0.4861 | val_loss 0.3174 | val_acc 85.5% | val_margin 2.2180
step 350: train_loss 0.4655 | val_loss 0.3020 | val_acc 88.4% | val_margin 2.0390
step 400: train_loss 0.4478 | val_loss 0.2959 | val_acc 84.8% | val_margin 2.5405


epoch 2/2:   0%|          | 0/6807 [00:00<?, ?it/s]

step 450: train_loss 0.2536 | val_loss 0.2854 | val_acc 87.0% | val_margin 2.4461
step 500: train_loss 0.2486 | val_loss 0.2769 | val_acc 87.0% | val_margin 2.5857
step 550: train_loss 0.2439 | val_loss 0.2723 | val_acc 87.0% | val_margin 2.5256
step 600: train_loss 0.2357 | val_loss 0.2679 | val_acc 86.2% | val_margin 2.7499
step 650: train_loss 0.2345 | val_loss 0.2577 | val_acc 87.0% | val_margin 2.8726
step 700: train_loss 0.2336 | val_loss 0.2531 | val_acc 88.4% | val_margin 2.9127
step 750: train_loss 0.2341 | val_loss 0.2480 | val_acc 88.4% | val_margin 2.9060
step 800: train_loss 0.2305 | val_loss 0.2468 | val_acc 88.4% | val_margin 2.9563
step 850: train_loss 0.2283 | val_loss 0.2474 | val_acc 88.4% | val_margin 3.0052
--- DPO training complete ---
Final val accuracy: 88.4%
Best val accuracy seen: 88.4%


## 9. Sanity check: generate on a few held-out prompts

Quick qualitative check before pushing - DPO shouldn't drastically change *what* the model can
do, just *which* of its plausible completions it prefers. If outputs look degenerate (repeating
loops, empty code, or truncated one-liners), your `BETA` is probably too low or `LEARNING_RATE`
too high for this dataset size - try `BETA=0.3` and/or `LEARNING_RATE=1e-6` first.

In [ ]:
policy_model.load_state_dict(torch.load(best_model_path, map_location=device))
policy_model.eval()

def generate_code(prompt_text, max_new_tokens=200, temperature=0.2, top_k=50):
    ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = policy_model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0].tolist())

for r in val_records[:3]:
    prompt_text = format_prompt(r["prompt"])
    generated = generate_code(prompt_text)
    print("=" * 80)
    print(prompt_text[:300])
    print("-" * 40, "generated", "-" * 40)
    print(generated[len(prompt_text):][:400])


### Problem
Can you create a Python dictionary with five key-value pairs, where the values are all prime numbers? However, the keys must follow the following criteria:
- The first key must be a palindrome and represent a type of animal that is often associated with intelligence.
- The second key mus
---------------------------------------- generated ----------------------------------------
import math
import random
import string
from collections import Counter
def is_prime(num):
    """Check if a number is prime."""
    if num <= 1:
        return False
    if num <= 3:
        return True
    if num % 2 == 0 or num % 3 == 0:
        return False
    i = 5
    while i * i <= num:
        if num % i == 0 or num % (i + 2) == 0:
            return False
        i += 6
    return True
d
### Problem
Write a Python code to find the top k largest numbers in a list, where k is a given positive integer. The list can contain duplicates and can have a length of up to 10^6 elements. The code shoul

In [ ]:
test_prompts = [
    "Write a function that checks if a string is a palindrome, ignoring spaces and case.",
    "Write a function to find the second largest number in a list of integers.",
    "Write a function that merges two sorted lists into one sorted list.",
    "Write a function to count the frequency of each word in a given string.",
    "Write a function that returns True if a number is a perfect square, False otherwise.",
]

def generate_with(model, prompt_text, max_new_tokens=220, temperature=0.2, top_k=50):
    ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        out = model.generate(
            idx, max_new_tokens=max_new_tokens,
            temperature=temperature, top_k=top_k,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0].tolist())[len(prompt_text):]

for q in test_prompts:
    prompt_text = format_prompt(q)
    dpo_out = generate_with(policy_model, prompt_text)
    ref_out  = generate_with(ref_model, prompt_text)

    print("=" * 90)
    print("PROMPT:", q)
    print("-" * 40, "BEFORE DPO (reference/SFT)", "-" * 40)
    print(ref_out[:500])
    print("-" * 40, "AFTER DPO (policy)", "-" * 40)
    print(dpo_out[:500])

PROMPT: Write a function that checks if a string is a palindrome, ignoring spaces and case.
---------------------------------------- BEFORE DPO (reference/SFT) ----------------------------------------
def is_palindrome(s):
    s = s.replace(" ", "").lower()
    return s == s[::-1]
```<｜end▁of▁sentence｜>
---------------------------------------- AFTER DPO (policy) ----------------------------------------
import string
def is_palindrome(s):
    s = s.lower()
    s = s.translate(str.maketrans('', '', string.punctuation))
    s = s.translate(str.maketrans('', '', string.punctuation))
    s = s.lower()
    s = s.translate(str.punctuation)
    s = s.translate(str.punctuation)
    s = s.translate(str.punctuation)
    s = s.lower()
    s = s.translate(str.punctuation)
    s = s.lower()
    s = s.translate(str.punctuation)
    s = s.translate(str.punctuation)
    s = s.lower()
    s = s.translate(str.pu
PROMPT: Write a function to find the second largest number in a list of integers.
-----------

## 10. Push the DPO model to the Hugging Face Hub

In [ ]:
from safetensors.torch import save_file
from huggingface_hub import HfApi, create_repo
from dataclasses import asdict
import os

policy_model.load_state_dict(torch.load(best_model_path, map_location=device))
policy_model.eval()

os.makedirs("hf_dpo_upload", exist_ok=True)
state_dict = {k: v.contiguous().clone() for k, v in policy_model.state_dict().items()}
save_file(state_dict, "hf_dpo_upload/model.safetensors")

with open("hf_dpo_upload/config.json", "w") as f:
    json.dump(asdict(config), f, indent=2)

tokenizer.save_pretrained("hf_dpo_upload")

readme = f'''---
license: apache-2.0
tags:
  - nanoGPT
  - code-generation
  - dpo
  - python
base_model: {HF_SFT_REPO}
datasets:
  - jondurbin/py-dpo-v0.1
---

# {HF_DPO_REPO.split("/")[-1]}

DPO fine-tune of [`{HF_SFT_REPO}`](https://huggingface.co/{HF_SFT_REPO}) on
[`jondurbin/py-dpo-v0.1`](https://huggingface.co/datasets/jondurbin/py-dpo-v0.1), a Python-only
preference dataset (chosen = tested/validated code, rejected = lower-quality generations).

- Beta: {BETA}
- Learning rate: {LEARNING_RATE}
- Held-out DPO accuracy (prefers chosen over rejected): {best_val_acc:.1%}
- Block size: {config.block_size}
- Vocab size: {config.vocab_size}

## Loading

This is a **custom architecture**, not a native `transformers` model:

```python
import json, torch
from safetensors.torch import load_file

with open("config.json") as f:
    cfg = json.load(f)

config = GPTConfig(**cfg)
model = GPT(config)
state_dict = load_file("model.safetensors")
model.load_state_dict(state_dict)
model.eval()
```
'''
import json as _json2
with open("hf_dpo_upload/README.md", "w") as f:
    f.write(readme)

create_repo(HF_DPO_REPO, token=HF_TOKEN, exist_ok=True)
api = HfApi()
api.upload_folder(folder_path="hf_dpo_upload", repo_id=HF_DPO_REPO, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HF_DPO_REPO}")


In [ ]:
from datasets import load_dataset

ds = load_dataset("jondurbin/py-dpo-v0.1", split="train")

print(ds)                      # shows num_rows and column names
print("Columns:", ds.column_names)
print()

# Look at the raw structure of a few rows
for i in range(3):
    row = ds[i]
    print("=" * 90)
    print(f"--- ROW {i} ---")
    print("id:", row["id"])
    print()
    print("PROMPT:")
    print(row["prompt"])
    print()
    print("CHOSEN:")
    print(row["chosen"])
    print()
    print("REJECTED:")
    print(row["rejected"])
    print()

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'id'],
    num_rows: 9466
})
Columns: ['prompt', 'chosen', 'rejected', 'id']

--- ROW 0 ---
id: 8c94f83f-6a5a-5f8c-98a2-e242d7764938

PROMPT:
Use the function to debug the given program and prevent the segmentation fault. Your solution should also handle the case where the array contains duplicate elements. You are not allowed to use any additional data structures. Additionally, the time complexity of your solution should be O(n) and the space complexity should be O(1).

```python
def debug_program(arr):
    n = len(arr)
    for i in range(n):
        if arr[i] == i:
            return i
    return -1

# Test Case
arr = [0, 1, 2, 3, 4]
print(debug_program(arr))  # Expected output: -1
```

**Additional Requirements:**

- The program should be able to handle arrays of any length.
- The program should be able to handle arrays with duplicate elements.
- The solution should use a divide and conquer approach to solve the problem.
- The

In [ ]:
import re
import statistics
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

# ============================================================================
# CONFIG - fill these in
# ============================================================================
HF_TOKEN = ""                                      # your NEW token (rotate the old one first)
OUTPUT_REPO = "Ananda100/py-dpo-code-only"          # where the cleaned dataset will be pushed
MAKE_PRIVATE = True
BLOCK_SIZE = 512                                     # must match your model's config.block_size
# ============================================================================

# ----------------------------------------------------------------------------
# 1. Load raw dataset
# ----------------------------------------------------------------------------
print("Loading jondurbin/py-dpo-v0.1 ...")
ds = load_dataset("jondurbin/py-dpo-v0.1", split="train")
print(ds)
print()

# ----------------------------------------------------------------------------
# 2. Code extraction
# ----------------------------------------------------------------------------
def extract_code_only(text):
    """Pull code out of a chosen/rejected field that may be prose + fenced code,
    or bare code with no fence at all. Returns None if nothing usable is found."""
    blocks = re.findall(r"```(?:python)?\s*\n?(.*?)```", text, flags=re.DOTALL)
    if blocks:
        code = max(blocks, key=len).strip()
        return code if code else None
    stripped = text.strip()
    if any(tok in stripped for tok in ("def ", "class ", "import ", "return ", "for ", "while ")):
        return stripped
    return None

# ----------------------------------------------------------------------------
# 3. Build cleaned rows
# ----------------------------------------------------------------------------
cleaned = []
skipped_no_code = 0
skipped_identical = 0

for row in ds:
    chosen_code = extract_code_only(row["chosen"])
    rejected_code = extract_code_only(row["rejected"])

    if chosen_code is None or rejected_code is None:
        skipped_no_code += 1
        continue

    if chosen_code.strip() == rejected_code.strip():
        skipped_identical += 1
        continue

    cleaned.append({
        "id": row["id"],
        "prompt": row["prompt"],
        "chosen": chosen_code,
        "rejected": rejected_code,
    })

print(f"After code extraction: kept {len(cleaned)} / {len(ds)} rows")
print(f"  skipped (no extractable code): {skipped_no_code}")
print(f"  skipped (chosen == rejected after cleaning): {skipped_identical}")
print()

# ----------------------------------------------------------------------------
# 4. Tokenizer + formatting helpers (must match your DPO notebook's template)
# ----------------------------------------------------------------------------
print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base", trust_remote_code=True)

def format_prompt(problem):
    return f"### Problem\n{problem}\n\n### Solution\n```python\n"

def format_completion(code):
    return f"{code}\n```"

# ----------------------------------------------------------------------------
# 5. Drop rows that won't fit fully inside block_size (no silent truncation -
#    a truncated 'chosen' could be missing its return statement or ending,
#    which would teach the model the wrong thing)
# ----------------------------------------------------------------------------
def fits_in_block(row, block_size=BLOCK_SIZE):
    p_ids = tokenizer.encode(format_prompt(row["prompt"]), add_special_tokens=False)
    c_ids = tokenizer.encode(format_completion(row["chosen"]), add_special_tokens=False)
    r_ids = tokenizer.encode(format_completion(row["rejected"]), add_special_tokens=False)
    chosen_total = len(p_ids) + len(c_ids) + 1   # +1 for eos
    rejected_total = len(p_ids) + len(r_ids) + 1
    return chosen_total <= block_size and rejected_total <= block_size

print("Filtering rows that don't fit within block_size ...")
before = len(cleaned)
cleaned = [row for row in cleaned if fits_in_block(row)]
print(f"Dropped {before - len(cleaned)} rows exceeding block_size={BLOCK_SIZE}")
print(f"Final row count: {len(cleaned)}")
print()

# ----------------------------------------------------------------------------
# 6. Preview a couple of final rows
# ----------------------------------------------------------------------------
print("=" * 90)
print("PREVIEW: final cleaned + filtered rows")
for row in cleaned[:2]:
    print("=" * 90)
    print("PROMPT:\n", row["prompt"][:300], "...\n")
    print("CHOSEN (code only):\n", row["chosen"])
    print()
    print("REJECTED (code only):\n", row["rejected"])
print()

# ----------------------------------------------------------------------------
# 7. Build Dataset object and push to your HF repo
# ----------------------------------------------------------------------------
cleaned_ds = Dataset.from_list(cleaned)
print(cleaned_ds)

if not HF_TOKEN:
    print("\nHF_TOKEN is empty - skipping push. Fill it in above and rerun this cell to push.")
else:
    print(f"\nPushing to https://huggingface.co/datasets/{OUTPUT_REPO} (private={MAKE_PRIVATE}) ...")
    cleaned_ds.push_to_hub(OUTPUT_REPO, token=HF_TOKEN, private=MAKE_PRIVATE)
    print(f"Done: https://huggingface.co/datasets/{OUTPUT_REPO}")

Loading jondurbin/py-dpo-v0.1 ...
Dataset({
    features: ['prompt', 'chosen', 'rejected', 'id'],
    num_rows: 9466
})

After code extraction: kept 9249 / 9466 rows
  skipped (no extractable code): 146
  skipped (chosen == rejected after cleaning): 71

Loading tokenizer ...
Filtering rows that don't fit within block_size ...
Dropped 2304 rows exceeding block_size=512
Final row count: 6945

PREVIEW: final cleaned + filtered rows
PROMPT:
 Use the function to debug the given program and prevent the segmentation fault. Your solution should also handle the case where the array contains duplicate elements. You are not allowed to use any additional data structures. Additionally, the time complexity of your solution should be O(n) and the  ...

CHOSEN (code only):
 def debug_program(arr):
    n = len(arr)
    return binary_search(arr, 0, n - 1)

def binary_search(arr, start, end):
    if start > end:
        return -1
    
    mid = (start + end) // 2
    
    if arr[mid] == mid:
        retu

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpbxy0j2es.parquet    :  12%|#2        |  526kB / 4.30MB            

Done: https://huggingface.co/datasets/Ananda100/py-dpo-code-only


In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/openai_humaneval")
print(dataset["test"][0])

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

{'task_id': 'HumanEval/0', 'prompt': 'from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    """ Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    """\n', 'canonical_solution': '    for idx, elem in enumerate(numbers):\n        for idx2, elem2 in enumerate(numbers):\n            if idx != idx2:\n                distance = abs(elem - elem2)\n                if distance < threshold:\n                    return True\n\n    return False\n', 'test': "\n\nMETADATA = {\n    'author': 'jt',\n    'dataset': 'test'\n}\n\n\ndef check(candidate):\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False\n    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True\n    assert can

In [ ]:
import torch

model.eval()

results = []

for sample in dataset["test"]:

    prompt = sample["prompt"]

    ids = tokenizer.encode(prompt)
    ids = torch.tensor(ids, device=device).unsqueeze(0)

    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=256,
            temperature=0.2,
            top_k=50,
            eos_token_id=tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(
        out[0][ids.shape[1]:].tolist(),
        skip_special_tokens=True,
    )

    results.append({
        "task_id": sample["task_id"],
        "completion": completion,
    })